In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("merged_and_cleaned_all_data.parquet", engine="pyarrow")

In [3]:
df.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1
2,Potato and Fennel Soup Hodge,In a heavy saucepan cook diced fennel and oni...,"[{""name"": ""fennel bulb (sometimes called anise...",40.0,165.0,7.0,6.0,garlish,NaN,2
3,Mahi-Mahi in Tomato Olive Sauce,Heat oil in heavy skillet over -high heat. Ad...,"[{""name"": ""extra-virgin olive oil"", ""quantity""...",22.0,NaN,NaN,NaN,main dish,NaN,3
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4


In [4]:
pd.set_option('display.max_colwidth', None)
print(df.iloc[144831].Ingredients)

[{"name": "glazed doughnut", "quantity": 6.0, "unit": "piece"}, {"name": "butter", "quantity": 0.5, "unit": "cup"}, {"name": "cream cheese", "quantity": 8.0, "unit": "ounce"}, {"name": "powdered sugar", "quantity": 1.5, "unit": "cup"}, {"name": "pure vanilla extract", "quantity": 1.0, "unit": "teaspoon"}, {"name": "Cool Whip", "quantity": 1.5, "unit": "cup"}, {"name": "marshmallow fluff", "quantity": 1.0, "unit": "cup"}, {"name": "variou fruit for topping. just make sure you slice enough. if you use banana i recommend coating them with lemon juice to avoid browning. pick your favorite fruit combination!!", "quantity": 1.0, "unit": "piece"}]


In [5]:
import pandas as pd
import numpy as np
import ast
import json

def extract_ingredient_names(ingredient_data):
    # 1) NumPy array veya Pandas Series ise listeye çevir.
    if isinstance(ingredient_data, (np.ndarray, pd.Series)):
        if ingredient_data.size == 0 or pd.isna(ingredient_data).all():
            return []
        ingredient_data = ingredient_data.tolist()

    # 2) None veya scalar NaN ise
    if ingredient_data is None:
        return []
    if isinstance(ingredient_data, float) and np.isnan(ingredient_data):
        return []

    # 3) Eğer Python listesi ise
    if isinstance(ingredient_data, list):
        parsed_data = ingredient_data

    # 4) Eğer dict ise, liste içerisine al
    elif isinstance(ingredient_data, dict):
        parsed_data = [ingredient_data]

    # 5) Eğer string ise, önce literal_eval, sonra JSON parse dene
    elif isinstance(ingredient_data, str):
        try:
            parsed_data = ast.literal_eval(ingredient_data)
            if isinstance(parsed_data, dict):
                parsed_data = [parsed_data]
            elif not isinstance(parsed_data, list):
                return []
        except (SyntaxError, ValueError):
            try:
                parsed_data = json.loads(ingredient_data)
                if isinstance(parsed_data, dict):
                    parsed_data = [parsed_data]
                elif not isinstance(parsed_data, list):
                    return []
            except (json.JSONDecodeError, TypeError):
                return []
    else:
        return []

    # 6) Parse edilmiş verinin liste olup olmadığını kontrol et
    if not isinstance(parsed_data, list):
        return []

    # 7) Tüm ingredient'lerden "name" alanını al (quantity/unit kontrolü kaldırıldı)
    names = []
    for item in parsed_data:
        if isinstance(item, dict):
            name = item.get('name')
            if name:
                names.append(str(name))

    # 8) Ingredient isimlerini içeren listeyi döndür
    return names


In [6]:
ingredients = pd.DataFrame()
ingredients["Ingredients"] = df['Ingredients'].apply(extract_ingredient_names)

In [7]:
ingredients["ID"] = df["ID"]

In [8]:
import re
import string
import unicodedata

def on_temizleme(strings):
    """
    Transformation rules:

    1) Remove everything inside parentheses.
    2) Reduce multiple spaces to a single space.
    3) Remove trailing punctuation marks.
    4) Remove leading spaces and punctuation marks.

    4.1) If the character before or after "/" is a number 
         (e.g., "10/20", "10/", "/20"), remove the numbers and the '/' character completely.

    4.6) If "kg", "ounce", "inch", "oz", "cm", "mi", "pkg", "tbsp", "lb", "g", "ml", "n/a", or "pound"
         is not adjacent to a letter,
         remove the entire token (i.e., the word along with any attached non-whitespace characters)
         that contains the unit.
         (Bu kural, "millet" kelimesinden "mi" kısmını çıkarmayacak şekilde, kelime sınırlarını kullanır.)

    4.5) If '/', '/possibly', '/ possibly', 'and/or', or "or possibly" appears, 
         replace it with ' or '.

    4.7) If the word "or" appears at the beginning of a sentence 
         (including punctuation, etc.), remove it as well.

    4.8) If the character "x" is not adjacent to a letter, remove the "x" character.
         (Yani; bir kelimenin parçasıysa dokunulmaz.)

    (New) If a token contains a hyphenated "or" (e.g., "short-or") and after "or" 
          there are non-letter and non-whitespace characters, remove "-or" and those characters.

    4.9) Remove words containing numbers completely.
    
    (New) Remove articles "a", "an" and "the" completely.
    
    (New) Remove the standalone "to" token if it appears as its own word.
    
    (New) If two (or more) consecutive "or" tokens appear, remove the duplicates.
    
    (New) Remove any text between " -" and "- " (including the delimiters).

    5) Replace any characters that are not allowed (letters, numbers, '-', ''', ',') with a space.
    6) Again, reduce multiple spaces to a single space and trim leading/trailing spaces.
    
    (New) After all cleaning steps, perform specific word substitutions.
    
    (New) Also, after converting to lowercase, replace "almonds3/" with "almond".
    
    (New) After removing parentheses, also remove any content enclosed in double quotes.
    """

    allowed_chars = set(string.ascii_letters + string.digits + "-',")
    cleaned_list = []

    for s in strings:
        # İlk olarak gelen stringi lowercase yapıyoruz.
        s = s.lower()
        # Küçük harfe çevirdikten sonra ilk iş: "almonds3/" kelimesini "almond" ile değiştir.
        s = s.replace("almonds3/", "almond")
        s = s.replace("piece release aluminum foil/ piece of regular aluminum foil", "aluminum foil")
        s = s.replace("wide jar/container with lid", "jar")
        s = s.replace("one or more large baking sheets, covered with parchment or buttered", "baking sheet")
        s = s.replace("little bit of dried/fresh oregano and thyme", "fresh oregano and thyme")
        # Aksanlı harfleri ASCII eşdeğerlerine dönüştür (örneğin "jícama" -> "jicama", "orégano" -> "oregano")
        s = unicodedata.normalize('NFKD', s)
        s = s.encode('ascii', 'ignore').decode('ascii')
        
        # -------------------- Yeni Adım: Sayı içeren kelimeleri sil (kural 4.9) --------------------
        s = re.sub(r"\b\w*\d+\w*\b", "", s)
        
        # -------------------- 1) Delete inner field of Paranthesis --------------------
        s = re.sub(r"\(.*?\)", "", s)
        # -------------------- Yeni Adım: ')' karakterinden önceki tüm her şeyi sil (') karakter dahil) ---------------
        s = re.sub(r'^.*?\)', '', s)
        # -------------------- Yeni Adım: Çift tırnak ("") içerisindeki tüm içeriği sil ---------------
        s = re.sub(r'".*?"', '', s)
        
        # -------------------- 2) Delete more than one spaces -----------------
        s = re.sub(r"\s+", " ", s).strip()

        # -------------------- 3) Delete punctuation at the end ------- 
        while s and s[-1] in string.punctuation:
            s = s[:-1]

        # -------------------- 4) Delete space/punctuation at the beginning --------
        while s and (s[0].isspace() or s[0] in string.punctuation):
            s = s[1:]

        # -------------------- 4.1) Delete slash + number ------------------
        s = re.sub(r"\d+/\d+", "", s)  # Örn. "10/20"
        s = re.sub(r"\d+/", "", s)     # Örn. "10/"
        s = re.sub(r"/\d+", "", s)     # Örn. "/20"

        # -------------------- 4.6) Delete tokens containing units -----------------
        pattern_units = r'((?:^|\s))\S*?(?<![A-Za-z])(?:kg|ounce|inch|inches|oz|cm|mi|pkg|tbsp|lb|g|ml|n/a|pound|pounds|qt|ounces|plu|pc|pt|torn up|s & b|t|c|accompaniment:|lrg|tsp.|ingredients|tb|canning|whl|colors,)(?![A-Za-z])\S*(?=\s|$)'
        s = re.sub(pattern_units, '', s)

        # -------------------- 4.5) Replace '/possibly', 'and/or', and 'or possibly' with ' or ' -----------------
        s = re.sub(r'(?:/(\s*possibly)?|and/or|\bor possibly\b)', ' or ', s)

        # -------------------- 4.7) Delete "or" at the beginning -------------
        s = re.sub(r'^[^a-zA-Z0-9]*or\b', '', s, flags=re.IGNORECASE)

        # -------------------- 4.8) Delete "x" if it is not adjacent to a letter -------------
        s = re.sub(r'(?i)(?<![a-z])x(?![a-z])', '', s)
        
        # -------------------- New: Remove hyphenated 'or' and trailing non-letter/whitespace characters ----------- 
        s = re.sub(r'(?<=\S)-or\b[^a-zA-Z\s]*', '', s)

        # -------------------- Remove articles "a", "an" and "the" -----------------
        s = re.sub(r'\b(?:a|an|the)\b', '', s)
        
        # -------------------- Yeni Adım: Remove standalone "to" -----------------
        s = re.sub(r'\bto\b', '', s)
        
        # -------------------- Yeni Adım: Eğer makale kaldırılınca string başında tek başına "or" kalıyorsa kaldır ---------------
        s = re.sub(r'^\s*\bor\b\s*', '', s)
        
        # -------------------- Yeni Adım: Peş peşe iki veya daha fazla "or" varsa, yineleneni kaldır ---------------
        s = re.sub(r'\bor(?:\s+or)+\b', 'or', s)
        
        # -------------------- Yeni Adım: " -" ile "- " arasındaki kısmı (delimitörler dahil) kaldır ---------------
        s = re.sub(r'\s-\s.*?\s-\s', ' ', s)

        # -------------------- 5) Not allowed chars swapping with space ----
        temp = []
        for ch in s:
            if ch in allowed_chars or ch.isspace():
                temp.append(ch)
            else:
                temp.append(" ")
        new_s = "".join(temp)

        # -------------------- Yeni Adım: Eğer string "or " ile bitiyorsa, kaldır ---------------
        if new_s.endswith("or "):
            new_s = new_s[:-len("or ")]

        # -------------------- Yeni Adım: Başta ve sonda bulunan noktalama işaretlerini kaldır ---------------
        new_s = re.sub(r'^[' + re.escape(string.punctuation) + r']+', '', new_s)
        new_s = re.sub(r'[' + re.escape(string.punctuation) + r']+$', '', new_s)

        # -------------------- 6) Delete more than one spaces ----------- 
        new_s = re.sub(r"\s+", " ", new_s).strip()

        # -------------------- Yeni Adım: Apostrof (') öncesindeki boşluğu kaldır ---------------
        new_s = re.sub(r"\s+'", "'", new_s)
        
        # -------------------- Yeni Adım: Belirtilen kelime düzeltmeleri (örnek düzeltmeler) ---------------
        new_s = re.sub(r'\bfrzn\b', 'frozen', new_s)
        new_s = re.sub(r'\bbsil\b', 'basil', new_s)
        new_s = re.sub(r'\bmangoe\b', 'mango', new_s)
        new_s = re.sub(r'\bstrawberrie\b', 'strawberries', new_s)
        new_s = re.sub(r'\btvanilla\b', 'vanilla', new_s)
        new_s = re.sub(r'\bbittersweet\b', 'bitter', new_s)
        
        # -------------------- Yeni Adım: Eğer "--" varsa, "-" ile değiştir ---------------
        new_s = new_s.replace("--", "-")
        
        # -------------------- Yeni Adım: Virgülden sonra boşluk yoksa ekle ---------------
        new_s = re.sub(r",(\S)", r", \1", new_s)

        # -------------------- Yeni Adım: Belirtilen kelime değişimlerini uygula ---------------
        replacements = {
            r'\barticoke\b': 'artichoke',
            r'\b(?:arugola|arugala)\b': 'arugula',
            r'\b(?:asparapu|asparaga)\b': 'asparagu',
            r'\b(?:advocado|avoocado|avokado|avocade)\b': 'avocado',
            r'\bshot\b': 'shoot',
            r'\b(?:broc|brocoli)\b': 'broccoli',
            r'\bbruss\b': 'brussel sprout',
            r'\b(?:cabagge|cabage)\b': 'cabbage',
            r'\b(?:carrorot|carrto|carrrot|carot)\b': 'carrot',
            r'\b(?:caulfilower|cauloflower|caulliflower|cauli)\b': 'cauliflower',
            r'\bceleriac\b': 'celery',
            r'\bchicorie\b': 'chicory',
            r'\b(?:cucumer|cucumbe|cucucmber|cucmber)\b': 'cucumber',
            r'\b(?:garllic|garli)\b': 'garlic',
            r'\b(?:letuc|lettu|lettice|letteuce|lettece)\b': 'lettuce',
            r'\bromain\b': 'romaine',
            r'\b(?:shroom|muhroom|mush)\b': 'mushroom',
            r'\b(?:onoin|onlon|onio)\b': 'onion',
            r'\b(?:scaleon|scallon|scalion)\b': 'scallion',
            r'\b(?:potaote|potaotoe|potaoe|potat)\b': 'potato',
            r'\b(?:rutebaga|rutabega|rutabag)\b': 'rutabaga',
            r'\b(?:shalott|shalot|shalllot|shallet|shal lot)\b': 'shallot',
            r'\b(?:spinnach|spini|spinatch)\b': 'spinach',
            r'\b(?:tomatillio|tomatilla)\b': 'tomatillo',
            r'\b(?:tomoatoe|tomaoe|toamtoe|tomatp|tomatio|tomata|tomate)\b': 'tomato',
            r'\bturnpi\b': 'turnip',
            r'\b(?:zuch|zucchi|zucci)\b': 'zucchini',
            r'\b(?:vegtable|vegitable|vegetible|vegeteble|vegetble|vegetaqble|vegetale|vegetalbe|vegetal)\b': 'vegetable',
            r'\b(?:chilly|chilli)\b': 'chili',
            r'\btamarin\b': 'tamarind',
            r'\b(?:apncot|appricot)\b': 'apricot',
            r'\b(?:blackbarrie|blackberrie)\b': 'blackberry',
            r'\bblueberrie\b': 'blueberry',
            r'\bboysenberrie\b': 'boysenberry',
            r'\bcherri\b': 'cherry',
            r'\b(?:limon|lemoon|lemom|lemmon)\b': 'lemon',
            r'\b(?:raspber|rasbe|rasp)\b': 'raspberry',
            r'\b(?:strawverrie|strawbrrie|strawbery|strawbertie|strawberrry|strawberrrie|strawberrir|strawb|strawberrie)\b': 'strawberry',
            r'\blingonberr\b': 'lingonberry',
            r'\bcranberrie\b': 'cranberry',
            r'\bannat\b': 'achiote',
            r'\belderberr\b': 'elderberry',
            r'\b(?:chiken|chichen|chick)\b': 'chicken',
            r'\b(?:prosciuttini|prosciutti|prosciuto)\b': 'prosciutto',
            r'\bsauage\b': 'sausage',
            r'\bobster\b': 'lobster',
            r'\b(?:shrump|shrmp|shrip|shrim)\b': 'shrimp',
            r'\bfilet\b': 'fillet',
            r'\bartic\b': 'arctic',
            r'\bsradine\b': 'sardine',
            r'\bcevice\b': 'ceviche',
            r'\b(?:youg|yogourt|joghurt|yoghurt)\b': 'yogurt',
            r'\bmoz\b': 'mozzarella',
            r'\btortilli\b': 'tortilla',
            r'\b(?:cibatta|ciabetta|ciabbata)\b': 'ciabatta',
            r'\bempenada\b': 'empanada',
            r'\b(?:spaghetinni|spagetti)\b': 'spaghetti',
            r'\b(?:kalamata oil|kalamata oive|kalamat olive)\b': 'kalamata olive',
            r'\b(?:parsey|parsl|parsely)\b': 'parsley',
            r'\bcinn\b': 'cinnamon',
            r'\bpeper\b': 'pepper',
            r"\b(?:birds eye|birdeye|birda eye chillie|bird chil|birdseye|bird s eye|bird's eye|bird eye|bird eye chili)\b": "bird's eye chili",
            r'\bceyenne\b': 'cayenne',
            r'\bsause\b': 'sauce',
            r'\bsoysause\b': 'soy sauce',
            r'\b(?:ketjap|kecap)\b': 'ketchup',
            r'\b(?:vinger|vinigar|vineger|viniger|vingegar|vingear|vingar|vinergar|vinegear|vinagreta|vinager|vinagar|venigar|veneger|vinega)\b': 'vinegar',
            r'\b(?:bbq|barbeque|barbicue|barbq|bq)\b': 'barbecue',
            r'\b(?:maynonaise|mayo|mayonnaise)\b': 'mayonaisse',
            r'\bwasabe\b': 'wasabi',
            r'\b(?:worstershire|worc)\b': 'worcestershire',
            r'\b(?:suger|sigar|sguar|surgar|suggar|sugauar|sugat|sugaar|sudar|suar|suagr)\b': 'sugar',
            r'\b(?:gel|galatin)\b': 'gelatin',
            r'\b(?:marsmallow|marsh|mashmellow|mashmallow|marhsmallow)\b': 'marshmallow',
            r'\bjell\b': 'jelly',
            r'\bbroiche\b': 'brioche',
            r'\breece\b': 'reese',
            r'\bcandi\b': 'candy',
            r'\badvocat\b': 'advocaat',
            r'\babsente\b': 'absinthe',
            r'\brhum\b': 'rum',
            r'\btequil\b': 'tequila',
            r'\bliquer\b': 'liqueur',
            r'\bliquor\b': 'liqueur',
            r'\bamareto\b': 'amaretto',
            r'\bamaretti\b': 'amaretto',
            r'\bmariner\b': 'marnier',
            r'\bbambo\b': 'bamboo',
            r'\b(?:bisqui|bisqick|biscut)\b': 'biscuit',
            r'\bveg\b': 'vegan',
            r'\bvegitarian\b': 'vegetarian',
            r'\btostido\b': 'tostito',
            r'\b(?:pep|peppet|peeper|paper)\b': 'pepper',
            r'\bcinamon\b': 'cinammon',
            r'\bcimmamon\b': 'cinammon',
            r'\b(?:cannola|cannoli|canoil|canol a|canol\'a|canol-a|canoloa|canoli)\b': 'canola',
            r'\b(?:oilve|oilive|ollive|olvice|olve|olvie)\b': 'olive',
            r'\bcarmel\b': 'caramel',
            r'\bcarmal\b': 'caramel',
            r'\bmargrine\b': 'margarine',
            r'\bmargirine\b': 'margarine',
            r'\bmargerine\b': 'margarine',
            r'\bmargar\b': 'margarine',
            r'\baburage\b': 'abura-age',
            r'\babura age\b': 'abura-age',
            r'\baburaage\b': 'abura-age',
            r'\bpow\b': 'powder',
            r'\beach thyme fresh\b': 'thyme',
        }
        for pattern, repl in replacements.items():
            new_s = re.sub(pattern, repl, new_s)
        
        cleaned_list.append(new_s)

    return cleaned_list


In [9]:
ingredients["cleaned_ingredients"] = ingredients["Ingredients"].apply(on_temizleme)

In [10]:
ingredients.head()

,Ingredients,ID,cleaned_ingredients
0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, Freshly ground black pepper to taste, whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas, turkey breast, thinly sliced, Bibb lettuce]",0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried french green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, freshly ground black pepper taste, whole-wheat lavash, cut in half crosswise, or flour tortillas, turkey breast, thinly sliced, bibb lettuce]"
1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, Pinch of dried thyme, crumbled, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into 1-inch chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, minced, Lettuce leaves, Cracked peppercorns, Minced fresh parsley, Bay leaves, French bread baguette slices, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or 3/4 teaspoon dried, crumbled, sugar]",1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, pinch of dried thyme, crumbled, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into chunks, well chilled, eggs, all purpose flour, tawny port, dried currants, minced, lettuce leaves, cracked peppercorns, minced fresh parsley, bay leaves, french bread baguette slices, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or teaspoon dried, crumbled, sugar]"
2,"[fennel bulb (sometimes called anise), stalks discarded, bulb cut into 1/2-inch dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet (baking) potatoes, chicken broth, milk]",2,"[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet potatoes, chicken broth, milk]"
3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, 14 1/2-ounce cans diced tomatoes with garlic, basil, and oregano in juice, 6-ounce mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, (packed) finely grated orange peel, Country-style white bread cut into 1/2-inch-thick slices, toasted]",3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, cans diced tomatoes with garlic, basil, and oregano in juice, mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, finely grated orange peel, country-style white bread cut into slices, toasted]"
4,"[12-ounce package frozen spinach soufflé, thawed, extra-wide egg noodles, freshly cooked, sour cream, purchased pesto sauce, ground nutmeg, grated sharp cheddar cheese]",4,"[package frozen spinach souffle, thawed, extra-wide egg noodles, freshly cooked, sour cream, purchased pesto sauce, ground nutmeg, grated sharp cheddar cheese]"


In [11]:
import re
import string

def process_list(ingredient_list):
    """Her satırdaki listeyi işler ve iç içe listeler halinde temizlenmiş kelimeler döndürür."""
    if not isinstance(ingredient_list, list):
        return []  

    processed_words = []
    for ingredient in ingredient_list:
        if not isinstance(ingredient, str):
            continue  

        split_ingredients = ingredient.split(" or ")

        cleaned_sublist = []
        for item in split_ingredients:
            # İstenmeyen karakterleri boşlukla değiştir.
            item = re.sub(r"[^,\w\s'\-]", " ", item)
            item = re.sub(r"\s+", " ", item).strip()
            
            # Eğer parçanın sonunda noktalama işareti varsa kaldır.
            item = item.rstrip(string.punctuation)
            
            if item:  
                cleaned_sublist.append(item)

        processed_words.append(cleaned_sublist)

    return processed_words

In [12]:
ingredients["splitted_ingredients"] = ingredients["cleaned_ingredients"].apply(process_list)

In [13]:
ingredients.head()

,Ingredients,ID,cleaned_ingredients,splitted_ingredients
0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried French green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, Fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, Freshly ground black pepper to taste, whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas, turkey breast, thinly sliced, Bibb lettuce]",0,"[low-sodium vegetable or chicken stock, dried brown lentils, dried french green lentils, celery, chopped, carrot, peeled and chopped, fresh thyme, kosher salt, tomato, cored, seeded, and diced, fuji apple, cored and diced, freshly squeezed lemon juice, extra-virgin olive oil, freshly ground black pepper taste, whole-wheat lavash, cut in half crosswise, or flour tortillas, turkey breast, thinly sliced, bibb lettuce]","[[low-sodium vegetable, chicken stock], [dried brown lentils], [dried french green lentils], [celery, chopped], [carrot, peeled and chopped], [fresh thyme], [kosher salt], [tomato, cored, seeded, and diced], [fuji apple, cored and diced], [freshly squeezed lemon juice], [extra-virgin olive oil], [freshly ground black pepper taste], [whole-wheat lavash, cut in half crosswise, flour tortillas], [turkey breast, thinly sliced], [bibb lettuce]]"
1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, Pinch of dried thyme, crumbled, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into 1-inch chunks, well chilled, eggs, all purpose flour, tawny Port, dried currants, minced, Lettuce leaves, Cracked peppercorns, Minced fresh parsley, Bay leaves, French bread baguette slices, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or 3/4 teaspoon dried, crumbled, sugar]",1,"[whipping cream, onions, chopped, salt, bay leaves, whole cloves, garlic clove, crushed, pepper, ground nutmeg, pinch of dried thyme, crumbled, shallots, minced, butter, trimmed boneless center pork loin, sinew removed cut into chunks, well chilled, eggs, all purpose flour, tawny port, dried currants, minced, lettuce leaves, cracked peppercorns, minced fresh parsley, bay leaves, french bread baguette slices, olive oil, red onions, halved, sliced, dried currants, red wine vinegar, canned chicken broth, chopped fresh thyme or teaspoon dried, crumbled, sugar]","[[whipping cream], [onions, chopped], [salt], [bay leaves], [whole cloves], [garlic clove, crushed], [pepper], [ground nutmeg], [pinch of dried thyme, crumbled], [shallots, minced], [butter], [trimmed boneless center pork loin, sinew removed cut into chunks, well chilled], [eggs], [all purpose flour], [tawny port], [dried currants, minced], [lettuce leaves], [cracked peppercorns], [minced fresh parsley], [bay leaves], [french bread baguette slices], [olive oil], [red onions, halved, sliced], [dried currants], [red wine vinegar], [canned chicken broth], [chopped fresh thyme, teaspoon dried, crumbled], [sugar]]"
2,"[fennel bulb (sometimes called anise), stalks discarded, bulb cut into 1/2-inch dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet (baking) potatoes, chicken broth, milk]",2,"[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish, onion, diced, unsalted butter, russet potatoes, chicken broth, milk]","[[fennel bulb , stalks discarded, bulb cut into dice, and feathery leaves reserved for garnish], [onion, diced], [unsalted butter], [russet potatoes], [chicken broth], [milk]]"
3,"[extra-virgin olive oil, chopped onion, dry white wine, anchovy paste, 14 1/2-ounce cans diced tomatoes with garlic, basil, and oregano in juice, 6-ounce mahi-mahi fillets, green olives, quartered, pitted, chopped fresh oregano, divided, (packed) finely grated orange peel, Country-style white bread cut into 1/2-

In [14]:
import pandas as pd

def extract_unique_ingredients_with_ids(df, ingredient_col="splitted_ingredients", id_col="ID", output_file="unique_ingredients.tsv"):
    ingredient_to_ids = {}

    for idx, row in df.iterrows():
        row_id = row[id_col]
        ingredients = row[ingredient_col]

        if not isinstance(ingredients, list):
            continue

        for sublist in ingredients:
            if not isinstance(sublist, list):
                continue  

            for ingredient in sublist:
                # Önce baştaki ve sondaki boşlukları temizleyelim.
                ingredient = ingredient.strip()
                # Eğer ingredient'in başında 'or ' varsa kaldırıyoruz (case-insensitive kontrol yapıyoruz).
                if ingredient.lower().startswith("or "):
                    ingredient = ingredient[3:].strip()
                
                if ingredient not in ingredient_to_ids:
                    ingredient_to_ids[ingredient] = set()  
                ingredient_to_ids[ingredient].add(row_id)

    output_df = pd.DataFrame([
        (ingredient, ", ".join(map(str, sorted(ids))))  
        for ingredient, ids in ingredient_to_ids.items()
    ], columns=["Ingredient", "IDs"])

    output_df.to_csv(output_file, sep="\t", index=False)


In [15]:
extract_unique_ingredients_with_ids(ingredients)

In [16]:
ingredients.to_csv('ingredients.tsv', sep='\t', index=False)